[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/06_error_analysis.ipynb)

# 06 · Error analysis (CPU)

Automatic failure taxonomy for every prediction file in `results/predictions`: invalid JSON, truncation, schema-invalid, wrong salary unit, missed/spurious skill, hallucinated field, location confusion, seniority mismatch, experience/role/company errors, with counts per approach and worked examples. Runs on CPU from saved predictions, locally or in Colab.

**How to run**
1. Runtime → Change runtime type → **CPU (no GPU needed)**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN (only to pull private artefacts)`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~1 min**. Cells after the setup are safe to re-run.

**Outputs:** `results/error_taxonomy.json` and `results/error_taxonomy.png`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    pass
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")


In [ ]:
# Optional: pull the latest GPU-run predictions from the HF artifacts repo.
PULL = False
if PULL:
    from jobx.hub import pull_artifacts
    pull_artifacts('.')

In [ ]:
import json
from pathlib import Path
import pandas as pd
from jobx.eval.run import SPLIT_FILES, parse_pred_name, score_file, load_gold
from jobx.eval.taxonomy import CATEGORIES, taxonomy_counts, worked_examples
scores, golds = {}, {}
for p in sorted(Path('results/predictions').glob('*__*.jsonl')):
    approach, split = parse_pred_name(p)
    if not Path(SPLIT_FILES.get(split, '-')).exists():
        continue
    scores[p.stem], _ = score_file(p)
    golds[p.stem] = load_gold(split)
counts = {k: taxonomy_counts(v) for k, v in scores.items()}
df = pd.DataFrame(counts).T
rates = (df[list(CATEGORIES)].div(df['n_examples'], axis=0) * 100).round(1)
Path('results/error_taxonomy.json').write_text(json.dumps(counts, indent=2))
rates

Rates are % of examples with at least one error of that type (an example can have several).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
ax = rates.T.plot(kind='barh', figsize=(9, 6))
ax.set_xlabel('% of examples'); ax.invert_yaxis(); plt.tight_layout()
plt.savefig('results/error_taxonomy.png', dpi=120)

## Per-field exact match

In [ ]:
from jobx.eval.metrics import aggregate
pd.DataFrame({k: {f: round(100 * v['em'], 1) for f, v in aggregate(s, n_boot=0)['per_field'].items()}
              for k, s in scores.items()}).T

## Worked examples per category

In [ ]:
for name, s in scores.items():
    for cat in CATEGORIES:
        ex = worked_examples(s, golds[name], cat, k=1)
        if not ex:
            continue
        e = ex[0]
        print(f'=== {name} | {cat} | {e["id"]}')
        print(e['text'][:400].replace(chr(10), ' / '))
        for f, d in e['diffs'].items():
            print(f'   {f}: gold={d["gold"]!r} pred={d["pred"]!r}')